In [ ]:
import pandas as pd
import seaborn as sns


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") or ""



In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature = 0.0
)

In [ ]:
llm.invoke("Hello, what are you?")

In [ ]:
from datetime import datetime
from langchain.tools import tool
from rag import retrieve

tool_log = []   # one row per tool hit (re-running this cell empties it)

@tool
def access_information(query: str) -> str:
    """Search the policy documents and return the passages most relevant to the query."""
    text = retrieve(query)
    tool_log.append({
        "time": datetime.now().isoformat(timespec="seconds"),
        "query": query,
        "chars_returned": len(text),
        "top_chunk_preview": text[:150].replace("\n", " "),
    })
    return text

tools = [access_information]

model_with_tools = llm.bind_tools(tools)


In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator

class MessagesState(TypedDict):
    messages : Annotated[list[AnyMessage], operator.add]
    query: str
    llm_calls:int
    answer:str



In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

SYSTEM_PROMPT = (
    "You answer questions about the policy documents. "
    "Always call access_information to look up facts before answering. "
    "Write the search query as 2-6 keywords for the topic of the question; "
    "leave out filler words like 'guide', 'document' and 'application'. "
    "If the returned passages do not contain the answer, say you could not find it "
    "in the documents instead of searching again. "
    "Answer only from what it returns."
)

def llm_call(state: MessagesState) -> MessagesState:
    msgs = state["messages"] or [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=state["query"]),
    ]
    answer = model_with_tools.invoke(msgs)
    return {"messages": [answer], "llm_calls": state["llm_calls"] + 1}


In [ ]:
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode, tools_condition

graph = StateGraph(MessagesState)

graph.add_node("llm_call",llm_call)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START,"llm_call")
graph.add_conditional_edges("llm_call", tools_condition)  # -> "tools" or END
graph.add_edge("tools", "llm_call")
app = graph.compile()



In [ ]:
#q_1 = input("Tell me your query?")

In [ ]:
result = app.invoke({
    "messages" : [],
    "query": "What is the refund policy?",
    "llm_calls":0,
    "answer":""
})

In [ ]:
result

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
from rag import _vectorstore
def eval_list(ques,k,vector_name):
    """This is analyze the RAG answers to see if the answers are correct.
    returns Recall@k and MRR"""
    string=[]
    scored=[]
    docs=_vectorstore(vector_name).similarity_search_with_score(ques, k=k) 
    for d,score in docs:
        string+=[d.page_content]
        scored+=[score]
    return string,scored


In [ ]:
import re, unicodedata

def normalize(text):
    text = unicodedata.normalize("NFKC", text)   # fixes ligatures like "ﬁ" -> "fi"
    text = text.lower()
    text = re.sub(r"\s+", " ", text)             # line breaks and double spaces -> one space
    return text

In [ ]:
def first_hit_rank(item):
    evidence = normalize(item["evidence"])
    for rank, chunk in enumerate(item["chunks"], start=1):
        if evidence=="":
            return None
        if evidence in normalize(chunk):        # is the evidence inside this normalized chunk?
            return rank        # return the rank
    return None               # nothing matched

In [ ]:
from rag import _vectorstore
from eval_data import EVAL_SET
import copy
def eval_metrics(data,k,vector_name):
    eval_ret2=copy.deepcopy(data)
    answerable=['u01','u02','u03','u04','u05']
    for i in eval_ret2:
        chunks, scores = eval_list(i["question"],k,vector_name)
        i['chunks'] = chunks
        i['scores'] = scores
    recall={}
    mrr=0
    tot=0
    for index,q in enumerate(eval_ret2):
        rank=first_hit_rank(q)
        print(index, q["id"], rank)
        if (q['id'] not in answerable):
            tot+=1
            if rank in recall:
                p=recall[rank]
                recall[rank]=p+1    
            else:
                recall[rank]=1
            if rank!=None:
                mrr=mrr+(1/rank)
    print(mrr/tot)
    print(recall)
    return (mrr/tot,recall)

In [ ]:
"""count=1
s=0
for i in recall:
    if count in recall:
        s+=recall[count]
        recall[count]=s
    else:
        break
    count+=1"""


In [ ]:
def recall_at(recall, k):
    tot = sum(recall.values())
    return sum(count for rank, count in recall.items()
                if rank is not None and rank <= k) / tot

In [ ]:
from rag import _vectorstore
def eval_list(ques,k,vector_name):
    """This is analyze the RAG answers to see if the answers are correct.
    returns Recall@k and MRR"""
    string=[]
    scored=[]
    docs=_vectorstore(vector_name).similarity_search_with_score(ques, k=k) 
    for d,score in docs:
        string+=[d.page_content]
        scored+=[score]
    return string,scored

In [ ]:
from rag import _vectorstore
def eval_metrics(data,k,vector_name):
    eval_ret2=copy.deepcopy(data)
    answerable=['u01','u02','u03','u04','u05']
    for i in eval_ret2:
        chunks, scores = eval_list(i["question"],k,vector_name)
        i['chunks'] = chunks
        i['scores'] = scores
    recall={}
    mrr=0
    tot=0
    for index,q in enumerate(eval_ret2):
        rank=first_hit_rank(q)
        print(index, q["id"], rank)
        if (q['id'] not in answerable):
            tot+=1
            if rank in recall:
                p=recall[rank]
                recall[rank]=p+1    
            else:
                recall[rank]=1
            if rank!=None:
                mrr=mrr+(1/rank)
    print(mrr/tot)
    print(recall)
    return (mrr/tot,recall)

In [ ]:
mrr,recall=eval_metrics(EVAL_SET,50,"faiss_index")

In [ ]:
for i in recall:
    if i!= None:
        print(i,recall_at(recall,i))

In [ ]:
mrr,recall=eval_metrics(EVAL_SET,50,"faiss_index500")

In [ ]:
for i in recall:
    if i!= None:
        print(i,recall_at(recall,i))

## Retrieval eval with the agent's own queries
Send every question to the LLM, keep the search query it writes for `access_information`, and score retrieval with those queries.

In [ ]:
# 1. Question -> LLM -> the search query it writes for the tool (the tool is not executed here)
import pandas as pd

rows = []
for q in EVAL_SET:
    resp = model_with_tools.invoke([SystemMessage(content=SYSTEM_PROMPT),
                                    HumanMessage(content=q["question"])])
    calls = resp.tool_calls
    rows.append({
        "id": q["id"],
        "question": q["question"],
        "answerable": q["answerable"],
        "tool_called": len(calls) > 0,
        "generated_query": calls[0]["args"].get("query") if calls else None,
    })

queries_df = pd.DataFrame(rows)
queries_df.to_csv("agent_queries.csv", index=False)
queries_df


In [ ]:
# 2. Score retrieval using the generated queries instead of the raw questions
agent_queries = dict(zip(queries_df["id"], queries_df["generated_query"]))
print("no tool call for:", [q["id"] for q in EVAL_SET if not agent_queries.get(q["id"])])

# same eval items, but the text we search with is the agent's query
agent_data = [{**q, "question": agent_queries.get(q["id"]) or q["question"]} for q in EVAL_SET]

mrr, recall = eval_metrics(agent_data, 8, "faiss_index")
print("MRR", mrr)
{k: recall_at(recall, k) for k in (1, 2, 4, 8)}


## Full agent run with a tool log
Runs the real graph (LLM -> tool -> LLM) for every question. `tool_log` gets one row each time `access_information` is hit.

In [ ]:
# 3. Full graph per question; the tool logs every hit
import time
from groq import RateLimitError
from langgraph.errors import GraphRecursionError

PAUSE_SECONDS = 14   # pause between questions (Groq on_demand: ~7000 input tokens per minute)
MAX_STEPS = 6        # graph steps per question: 2 searches + the final answer (LLM call or tool call = 1 step)
MAX_TRIES = 3        # attempts per question when Groq says rate limited
RETRY_WAIT = 30      # seconds to wait before retrying a rate-limited question

tool_log.clear()     # clear in place so the tool keeps writing to the same list
runs = []
for q in EVAL_SET:
    if runs and runs[-1]["status"] == "daily_limit":     # no point continuing once the daily budget is gone
        print("Stopped: Groq daily token limit reached.", runs[-1]["error"])
        break
    t0 = time.time()
    for attempt in range(1, MAX_TRIES + 1):
        n0 = len(tool_log)
        status, answer, llm_calls, error = "ok", None, None, None
        try:
            result = app.invoke({"messages": [], "query": q["question"], "llm_calls": 0, "answer": ""},
                                {"recursion_limit": MAX_STEPS})
            answer, llm_calls = result["messages"][-1].content, result["llm_calls"]
        except GraphRecursionError:
            status = "hit_step_limit"                # the model kept calling the tool
        except RateLimitError as e:
            error = str(e)[:300]
            # "tokens per day" will not clear in 30 seconds, so do not retry it
            status = "daily_limit" if "per day" in error else "rate_limited"
        except Exception as e:
            status, error = f"error: {type(e).__name__}", str(e)[:300]
        if status != "rate_limited" or attempt == MAX_TRIES:
            break
        del tool_log[n0:]                            # drop this attempt's partial tool hits
        time.sleep(RETRY_WAIT)
    for row in tool_log[n0:]:                        # tag this question's tool hits
        row["id"], row["question"] = q["id"], q["question"]
    runs.append({"id": q["id"], "question": q["question"], "status": status, "attempts": attempt,
                 "llm_calls": llm_calls, "tool_hits": len(tool_log) - n0,
                 "final_answer": answer, "error": error})
    time.sleep(max(0, PAUSE_SECONDS - (time.time() - t0)))

runs_df = pd.DataFrame(runs)
tool_log_df = pd.DataFrame(tool_log)
runs_df.to_csv("agent_runs.csv", index=False)
tool_log_df.to_csv("tool_log.csv", index=False)
runs_df
